# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AndruMark/flyrank-ml-internship/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

### Task Type: Supervised Ranking & Calibrated Priority Scoring (Pointwise Learning-to-Rank)

While content decay can be framed as a binary classification problem ($Y \in \{0, 1\}$ for whether a page is declining), the actual operational problem is **ranking and priority scoring**.

Editorial teams operate under fixed capacity constraints (e.g., refreshing 10 to 20 pages per month per account). Simply classifying 1,000 pages as "declining" is unusable for a team that can only tackle 20. The system must output a continuous, exposure-weighted risk score:
$$\text{Priority Score} = \hat{P}(\text{is\_declining} = 1 \mid X) \times \text{Exposure Weight}$$

This transforms the task into an operational triage ranker: estimating the calibrated posterior probability of decay and sorting the candidate URLs so that the top-$K$ recommendations maximize intercepted traffic loss.

In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

### Target Definition and Proxy Formulation

* **Target Variable:** Binary decay indicator `is_declining_label` $\in \{0, 1\}$, derived from the trailing search performance trajectory (`trend_direction == 'down'`).
* **Underlying Proxy:** A page is labeled as declining if its recent 30-day search visibility (impressions/clicks) reflects a statistically meaningful drop compared to its baseline 90-day rolling average.
* **Why this is a Proxy (and its Nuances):**
  * True "content decay" is a latent quality (information obsolescence, competitor displacement, or search intent divergence). We cannot directly measure text obsolescence; we observe its downstream footprint in search telemetry.
  * Seasonal traffic dips or macro query fluctuations can produce false drops. Therefore, candidate pages must satisfy eligibility guardrails (`impressions_90d > 0` and `content_age_days >= 90`). Zero-impression URLs are explicitly excluded because trend metrics and division-based features (CTR, average rank) are mathematically undefined and reflect indexing issues rather than content decay.

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Success metric

*One metric you can defend. What number means 'good'?*

### Operational and Technical Success Metrics

* **Primary Business Metric: Precision@$K$ (where $K \in \{20, 50\}$)**
  $$\text{Precision@}K = \frac{1}{K} \sum_{i=1}^K \mathbb{I}(y_{(i)} = 1)$$
  Because editorial bandwidth is strictly bounded, the model is judged solely on the quality of its top recommendations. If an editor reviews the top 20 flagged URLs and 16 are genuinely decaying high-value assets, Precision@20 is 80%, ensuring writer time is never squandered.
* **Secondary Ranking Metric: NDCG@$K$ (Normalized Discounted Cumulative Gain)**
  Evaluates whether the steepest drops on the highest-traffic pages appear at the very top of the triage queue rather than buried at position 19 or 20.
* **Why Standard Metrics Fail Here:**
  * **Accuracy** is distorted by class imbalance and treats an error on a page with 10 impressions identically to an error on a page with 50,000 impressions.
  * **ROC-AUC** evaluates global ranking across the entire long tail of thousands of low-impact pages that editors will never inspect. Only top-tier precision matters.

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

In [4]:
import os
import pandas as pd
import numpy as np

# 1. Load starter dataset
data_path = os.path.join("..", "..", "data", "raw", "content_refresh_anonymized.csv")
if not os.path.exists(data_path):
    data_path = os.path.join("data", "raw", "content_refresh_anonymized.csv")

df_raw = pd.read_csv(data_path)

# 2. Filter cohort and construct target
# Exclude pages under 90 days (immature) and pages with 0 impressions (undefined CTR/trend)
df_clean = df_raw[(df_raw["impressions_90d"] > 0) & (df_raw["content_age_days"] >= 90)].copy()
df_clean["is_declining_label"] = df_clean["trend_direction"].str.lower().eq("down").astype(int)

# 3. Define feature and target schema
feature_cols = [
    "impressions_90d",
    "clicks_90d",
    "ctr",
    "avg_position",
    "content_age_days",
    "days_since_last_update"
]
metadata_cols = ["client_id", "position_tier", "content_type"]
target_col = "is_declining_label"

# 4. Display the concrete unit of analysis
unit_of_analysis_df = df_clean[metadata_cols + feature_cols + [target_col]].head(5)

print(f"=== UNIT OF ANALYSIS: One row = One unique URL over a 90-day performance window ===")
print(f"Filtered Cohort Size: {len(df_clean):,} eligible pages across {df_clean['client_id'].nunique()} client domains.")
print(f"Target Distribution: {df_clean[target_col].mean():.1%} declining vs {1 - df_clean[target_col].mean():.1%} healthy/growing.\n")
display(unit_of_analysis_df)


=== UNIT OF ANALYSIS: One row = One unique URL over a 90-day performance window ===
Filtered Cohort Size: 30,000 eligible pages across 32 client domains.
Target Distribution: 54.2% declining vs 45.8% healthy/growing.



,client_id,position_tier,content_type,impressions_90d,clicks_90d,ctr,avg_position,content_age_days,days_since_last_update,is_declining_label
0,client_f369cb89fc,striking,keyword article,3803,29,0.76,10.6,187,20,1
1,client_4e07408562,page_3_5,keyword article,15320,7,0.05,20.3,445,25,1
2,client_7f2253d7e2,page_3_5,keyword article,12581,11,0.09,36.5,141,20,1
3,client_19581e27de,page_1,keyword article,11751,58,0.49,6.2,463,22,0
4,client_3fdba35f04,page_3_5,keyword article,19140,24,0.13,44.0,263,14,1


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

### Why Machine Learning Beats Fixed Heuristics

1. **Non-Linear SERP Real Estate:**
   A fixed threshold like `days_since_update > 180 AND impressions > 500` is fundamentally rigid. A page dropping from rank 2 to rank 4 loses vastly more organic traffic than a page dropping from rank 22 to 24. A learned decision model captures non-linear split points where high ranking exposure magnifies small positional losses.
2. **Multivariate Trade-offs Without Hard Cutoffs:**
   Static rules suffer from boundary cliffs: an article 179 days old is ignored while a 181-day-old article is flagged. Machine learning smoothly integrates content age, baseline volume, CTR velocity, and position tiers into a unified probability distribution.
3. **Cross-Client Adaptability:**
   Traffic scales differ by orders of magnitude across domains. A hardcoded impression threshold that works for a high-traffic B2C publisher completely fails on a niche enterprise B2B catalog. Learned tree models identify relative decay signals that generalize across unseen client distributions.

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [X] Every section above is filled — markdown thinking AND the code that backs it
- [X] The notebook runs top to bottom with no errors (Runtime → Run all)
- [X] No client names, URLs, or private queries anywhere
- [X] My claims use careful words: observed, measured, directional, decision-support
- [X] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.